# Notebook 01d — PPO baseline on `MiniGrid-UnlockPickup-v0`

Same pipeline as `01_ppo_baseline`, pointed at UnlockPickup. This is **task 4
of the four** in the experiment plan's E1 ladder: DoorKey-6x6, Unlock,
RedBlueDoors-6x6, UnlockPickup.

**The task.** Pick up the key, unlock the door, then pick up the target box in
the locked room. Success is the final box pickup, not merely opening the door.

**What the web check says.** Farama registers `MiniGrid-UnlockPickup-v0`, gives
it 7 actions, and documents the reward as `1 − 0.9·step_count/max_steps`. The
episode terminates when the correct box is picked up or on timeout, and Farama
states the task can be solved without relying on language. I did not find a
specific rl-baselines3-zoo tuned row or pretrained PPO checkpoint for this env;
the nearest solved reference is `MiniGrid-Unlock-v0` with the shared MiniGrid PPO
recipe.

**Three env-specific things.**

1. `env.success_on: terminated` is safe for this env's success definition: the
   environment terminates on correct box pickup; timeout is `truncated`, not
   `terminated`.
2. The grid is **11×6, not square** (fully-observable obs 11×6×3 = 198). The
   visitation plot cannot infer that from the packed state, so `GRID_SHAPE` is
   passed explicitly.
3. The sub-goal probe runs in `key_door` mode, so `subgoal1` = carrying the key
   and `subgoal2` = door open. Unlike Unlock, `subgoal2` is **not** success here:
   the final leg is door open → target box picked up.

**What to expect.** Uniform-random success was **0/250** episodes in this repo's
env pool, so this is a harder baseline than Unlock. The 1M-frame budget is
headroom for a first PPO attempt; if it fails, read the ladder before tuning:
key flat, door flat, and solved flat are different failures.

Hyperparameters are this repo's MiniGrid CNN recipe, not a literal RL-Zoo copy.
RL-Zoo uses FlatObs + MlpPolicy over the egocentric observation; this project
uses FullyObs + ImgObs + CNN so the later counterfactual oracle has a Markov
state-to-network-input map.


---
## ▶ Knobs

In [1]:
# ----------------------------------------------------------------------------
# EDIT ME
# ----------------------------------------------------------------------------
ENV_CONFIG    = "unlockpickup_cf"    # config/envs/unlockpickup.yaml

FORCE_RETRAIN = True

# What subgoal1 / subgoal2 mean in THIS environment. Plot labels only.
# Success is the third rung: after the key and door, pick up the target box.
SUBGOAL_LABELS = ("Picked up the key", "Opened the door")

# The grid is 11x6 and the packed sim_state does not record its shape, so the
# visitation plot has to be told. plot_minigrid_visitation refuses to guess
# rather than silently mis-binning every visit.
GRID_SHAPE     = (11, 6)

# OVERRIDES = {
#     # "ppo.total_timesteps": 200_000,   # a smoke run before the full 1M
#     # "run.seeds": (0,),                # one seed while iterating
#     # "ppo.ent_coef": 0.0,              # the rl-baselines3-zoo MiniGrid value
#     # "ppo.learning_rate": 2.5e-4,      # the rl-baselines3-zoo MiniGrid value
#     # "ppo.n_epochs": 10,               # the rl-baselines3-zoo MiniGrid value
#     # "run.run_name": "unlockpickup_zoo_hp",
# }
OVERRIDES = {
    "env.layout_seeds": [0],
    "ppo.ent_mode": "fixed",
    "ppo.ent_coef": 0.1,
    "ppo.prob_floor_start": 0.20,
    "ppo.prob_floor_end": 0.05,
    "ppo.total_timesteps": 1_500_000,
    "run.run_name": "unlockpickup_fixed0_floor_stronger",
    "run.seeds": [0],
}
# ----------------------------------------------------------------------------


## 0. Setup

In [2]:
# Reload edited modules automatically. Note this still cannot add a field to
# an already-imported dataclass -- for config schema changes, restart the kernel.
%load_ext autoreload
%autoreload 2

import sys, pathlib, time

ROOT = pathlib.Path.cwd()
if not (ROOT / "config").is_dir():          # launched from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import make_config, list_env_configs, RUNS_DIR, FIGURES_DIR, seed_dir
from dataio import load_trajectories, validate, load_checkpoint, list_checkpoints
from envs.env_pool import make_env
from utils.logging import read_scalars
from utils.plotting import (
    plot_learning_curves, plot_episode_returns, plot_grid,
    plot_subgoal_ladder, plot_signal_diagnostics,
    plot_minigrid_progress, plot_action_usage, plot_minigrid_visitation,
    savefig,
)

print("project root:", ROOT)
print("configured environments:", sorted(list_env_configs()))

project root: /Users/charithapalika/Desktop/Personal Projects/PPO_CF
configured environments: ['cartpole', 'cartpole_cf', 'doorkey5x5', 'doorkey5x5_cf', 'doorkey6x6', 'doorkey6x6_cf', 'doorkey8x8', 'doorkey8x8_cf', 'keycorridors3r1', 'keycorridors3r1_cf', 'lavagap', 'lavagap_cf', 'mountaincar', 'mountaincar_cf', 'redbluedoors6x6', 'redbluedoors6x6_cf', 'redbluedoors8x8', 'redbluedoors8x8_cf', 'taxi', 'taxi_cf', 'unlock', 'unlock_cf', 'unlockpickup', 'unlockpickup_cf']


## 1. Configuration

In [3]:
cfg = make_config(ENV_CONFIG, **OVERRIDES)
RUN_NAME  = cfg.run.run_name
SEEDS     = tuple(cfg.run.seeds)
IS_MINIGRID = cfg.env.env_id.startswith("MiniGrid")

# The packed sim_state records no width/height and these grids are not
# square (6x6 -> 12x6, 8x8 -> 16x8), so read the shape from the env rather
# than hard-coding it -- plot_minigrid_visitation refuses to guess.
if GRID_SHAPE is None and IS_MINIGRID:
    _e = make_env(cfg.env.env_id, cfg.env.max_episode_steps,
                  fully_observable=cfg.env.fully_observable)
    _e.reset(seed=0)
    GRID_SHAPE = (_e.unwrapped.grid.width, _e.unwrapped.grid.height)
    _e.close()
    print(f"grid shape (W, H): {GRID_SHAPE}")

FIG = FIGURES_DIR / f"nb01_{RUN_NAME}"
FIG.mkdir(parents=True, exist_ok=True)

print(cfg.summary())

if IS_MINIGRID:
    from envs.minigrid_env import ACTION_NAMES
    print("\nactions:", ACTION_NAMES)
else:
    ACTION_NAMES = [str(i) for i in range(make_env(cfg.env.env_id).action_space.n)]

# Order-of-magnitude only; measured ~1,700 steps/s for the MiniGrid CNN config
# on the reference machine.
est = cfg.ppo.total_timesteps / 1700 / 60
print(f"\nrough runtime      ~{est:.0f} min/seed  ({est * len(SEEDS):.0f} min total)")

config             unlockpickup_cf
env                MiniGrid-UnlockPickup-v0  (n_envs=16, obs_norm=image)
layouts            1 fixed (cycle)
total_timesteps    1,500,000  per seed
rollout batch      2048  (16 envs x 128 steps)
updates            732
minibatch size     256  x 4 epochs
gamma / lambda     0.99 / 0.95
lr                 0.001  (anneal=False)
encoder            cnn  (shared=True)
entropy            fixed, coef 0.1
prob floor         0.2 -> 0.05  (0 = off)
adv norm           batch  (min_std 1e-06)
policy gradient    cf_all_action  alpha_gae=1.0 alpha_cf=0.5, Q_g horizon=64 x2 rollouts, subsample=5%, restore=fast
reward shaping     off
warm start         none
seeds              [0]
checkpoints at     ['10%', '30%', '50%', '75%', '100%']

actions: ['0 turn left', '1 turn right', '2 forward', '3 pickup', '4 drop', '5 toggle', '6 done']

rough runtime      ~15 min/seed  (15 min total)


## 2. Train

`run_seeds` writes, per seed, into `runs/<run_name>/seed_<n>/`:
`trajectories.npz`, `checkpoints/ckpt_{010,030,050,075,100}.pt`, `scalars.csv`,
`episodes.csv`, and `runs/<run_name>/config.json` — the exact configuration
used, so any result can be traced back to what produced it.

The same run from the shell, with no notebook involved:

```bash
python -m scripts.train --env doorkey5x5
python -m scripts.train --env doorkey5x5 --frames 200000 --run-name smoke5x5
python -m scripts.train --env doorkey8x8 --set ppo.ent_coef=0.02 env.layout_seeds=[0,1,2,3]
```

In [4]:
from scripts.train import run_seeds

already = all((seed_dir(RUN_NAME, s) / "scalars.csv").exists() for s in SEEDS)

if already and not FORCE_RETRAIN:
    print(f"found existing run at {RUNS_DIR / RUN_NAME} — skipping training")
    print("set FORCE_RETRAIN = True to re-run")
    results = None
else:
    t0 = time.time()
    results = run_seeds(cfg)
    print(f"\ntotal wall time: {(time.time() - t0) / 60:.1f} min")


=== seed 0 ====================================================


/opt/homebrew/anaconda3/envs/workbench/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


  oracle check: replay exact over 96 transitions (reward err 0, state err 0); centering max 1.25e-08
  upd    10/732  step   20,480  ret   0.000  succ 0.00  sg1 0.75 sg2 0.09  ent 1.936  advstd 1.26e-02  ev -1.147  175 sps
  upd    20/732  step   40,960  ret   0.005  succ 0.01  sg1 0.82 sg2 0.13  ent 1.936  advstd 8.44e-03  ev -0.112  174 sps
  upd    30/732  step   61,440  ret   0.000  succ 0.00  sg1 0.80 sg2 0.14  ent 1.922  advstd 1.17e-02  ev  0.736  174 sps
  upd    40/732  step   81,920  ret   0.001  succ 0.01  sg1 0.85 sg2 0.26  ent 1.926  advstd 5.84e-03  ev -1.077  174 sps
  upd    50/732  step  102,400  ret   0.002  succ 0.01  sg1 0.88 sg2 0.21  ent 1.929  advstd 4.20e-03  ev -0.569  174 sps
  upd    60/732  step  122,880  ret   0.002  succ 0.01  sg1 0.81 sg2 0.07  ent 1.920  advstd 2.34e-03  ev -0.309  172 sps
  upd    70/732  step  143,360  ret   0.000  succ 0.00  sg1 0.81 sg2 0.12  ent 1.918  advstd 2.50e-03  ev  0.162  171 sps
  upd    80/732  step  163,840  ret   0.003  

KeyboardInterrupt: 

## 3. Load everything back

In [ ]:
scalars  = {s: read_scalars(seed_dir(RUN_NAME, s) / "scalars.csv") for s in SEEDS}
episodes = {s: pd.read_csv(seed_dir(RUN_NAME, s) / "episodes.csv")  for s in SEEDS}
trajs    = ({s: load_trajectories(seed_dir(RUN_NAME, s) / "trajectories.npz") for s in SEEDS}
            if cfg.run.record_trajectories else {})

S0 = SEEDS[0]
summary = pd.DataFrame({
    "first_success_step": {s: (int(e.loc[e["success"], "global_step"].iloc[0])
                               if e["success"].any() else None) for s, e in episodes.items()},
    "episodes":           {s: len(e) for s, e in episodes.items()},
    "total_successes":    {s: int(e["success"].sum()) for s, e in episodes.items()},
    "final_success_100":  {s: d["success_rate_100"].iloc[-1] for s, d in scalars.items()},
    "final_return_100":   {s: d["mean_return_100"].iloc[-1] for s, d in scalars.items()},
    "final_entropy":      {s: d["entropy"].iloc[-1] for s, d in scalars.items()},
    "final_expl_var":     {s: d["explained_variance"].iloc[-1] for s, d in scalars.items()},
    "traj_rows":          {s: len(t) for s, t in trajs.items()} if trajs else {},
})
summary.index.name = "seed"
summary.round(4)

## 4. Return and success

UnlockPickup's reward is `1 − 0.9·(steps / max_steps)` on success and `0`
otherwise. Success is picking up the target box after unlocking the door, so
mean return measures both solving and solving quickly. `max_steps` is 288 here;
an episode length pinned near that limit means the policy is mostly timing out.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_episode_returns(episodes, window=50, ax=axes[0])
plot_learning_curves(scalars, y="success_rate_100",
                     ylabel="success rate (last 100 ep)", title="Success rate", ax=axes[1])
axes[1].set_ylim(-0.05, 1.05)
plot_learning_curves(scalars, y="mean_length_100",
                     ylabel="episode length", title="Episode length", ax=axes[2])
fig.tight_layout()
savefig(fig, FIG / "return_and_success.png")
plt.show()

## 5. The sub-goal ladder — the metric that matters on a sparse task

Success rate alone is a poor progress signal: it can sit at exactly zero while
the policy is either improving or dying, and the two look identical. UnlockPickup
has a three-step behavior chain, but the current probe exposes the first two
hard prerequisites:

| what you see | what it means | what to do |
|---|---|---|
| all three flat | no learning signal at all | check `adv_std_raw` below first |
| key rises, door flat | stuck at the locked door | check `π(toggle)` and whether key pickup collapses |
| door rises, success flat | reaches the locked room but does not finish | check `π(drop)` and final `π(pickup)` |
| everything rises then collapses | rare reward averaged away | lower `ent_coef`, or try `ent_mode: adaptive` |

On this env the gap between `subgoal2` and success is the new signal: opening the
door is not enough, because the agent still has to pick up the target box.


In [ ]:
fig = plot_subgoal_ladder(scalars, labels=SUBGOAL_LABELS)
fig.suptitle(f"{RUN_NAME} — sub-goal ladder", y=1.04, fontsize=11)
savefig(fig, FIG / "subgoal_ladder.png")
plt.show()

# The two logged sub-goals are latched per episode by EnvPool:
# subgoal1 = ever carried the key, subgoal2 = ever opened the door. Success is
# the later target-box pickup. This printout keeps the three rungs visible even
# before looking at the plots.
for s in SEEDS:
    e = episodes[s]
    sg1 = e["subgoal1"].astype(bool) if "subgoal1" in e.columns else None
    sg2 = e["subgoal2"].astype(bool) if "subgoal2" in e.columns else None
    print(f"seed {s}: {len(e)} episodes | success {int(e['success'].sum())}")
    if sg1 is not None:
        print(f"  picked up key: {int(sg1.sum())} ({sg1.mean():.1%})")
    if sg2 is not None:
        print(f"  opened door:   {int(sg2.sum())} ({sg2.mean():.1%})")


## 6. Was there a signal at all?

`adv_std_raw` is the spread of the advantages **before** whitening. It separates
the two situations every other diagnostic confounds: *learning slowly* versus
*receiving nothing to learn from*.

If it sits at the `norm_adv_min_std` floor (1e-6), the rollout carried no
signal, and any movement in entropy, `approx_kl` or `clipfrac` during that
stretch is numerical noise — not exploration. The 3M DoorKey-8x8 run held
`clipfrac` at 0.05–0.18 while `v_loss` was 1e-19: every one of those updates was
the policy being random-walked by rescaled float error, which is exactly what
`PPOTrainer._normalise` now refuses to do.

In [ ]:
fig = plot_signal_diagnostics(scalars)
fig.suptitle(f"{RUN_NAME} — is there a signal?", y=1.04, fontsize=11)
savefig(fig, FIG / "signal.png")
plt.show()

for s, d in scalars.items():
    frac_dead = float((d["adv_std_raw"] < 10 * cfg.ppo.norm_adv_min_std).mean())
    print(f"  seed {s}: {frac_dead:.1%} of logged updates had effectively no advantage signal")

## 7. Optimisation diagnostics

What to look for:

- **entropy** — starts near log K (log 7 = 1.946 on MiniGrid). Pinned there means
  the policy is committing to nothing; collapsing toward 0 *before* any success
  means the run is dead (how the MountainCar and Taxi attempts failed).
- **explained_variance** — the critic's quality. NB02's oracle is built directly
  on V, so a low value here caps everything downstream.
- **approx_kl / clipfrac** — update size. Read them together with `adv_std_raw`:
  a healthy-looking clipfrac on a zero-signal rollout means nothing.
- **grad_norm_actor vs grad_norm_critic** — logged separately because on
  MountainCar the critic's gradient swamped a single global clip and silently
  froze the policy. Only populated when `separate_grad_clip` is in effect (it is
  disabled automatically when `share_encoder` is on).

In [ ]:
wanted = ["entropy", "approx_kl", "clipfrac", "adv_std_raw",
          "pg_loss", "v_loss", "explained_variance",
          "grad_norm_actor", "grad_norm_critic", "ent_coef", "prob_floor",
          "lr", "sps", "n_episodes"]
keys = [k for k in wanted
        if all(k in d.columns and np.isfinite(d[k]).any() for d in scalars.values())]
skipped = [k for k in wanted if k not in keys]
if skipped:
    print("no data (expected for the inactive controllers):", skipped)

fig = plot_grid(scalars, keys, ncols=3, figsize=(15, 4 * ((len(keys) + 2) // 3)))
savefig(fig, FIG / "diagnostics.png")
plt.show()

## 8. Per-action behaviour

The single most important diagnostic learned from the Taxi attempt. There the
policy drove `π(PICKUP)` to 0.0016 and `π(DROPOFF)` below 0.007 — making success
impossible — while total entropy still read a healthy 1.23 of log 6. **Every
aggregate metric looked fine.** Entropy is maximised just as well by spreading
mass over the movement actions, so only a per-action view reveals it.

UnlockPickup needs `pickup` (3), `toggle` (5), and probably `drop` (4). The key
opens the locked door but is not consumed by the upstream `Door.toggle` code,
and MiniGrid's generic pickup code only picks up the target box when
`carrying is None`. If any of those action probabilities collapses toward zero,
the run can stall even when aggregate entropy looks healthy — that is what
`ppo.prob_floor_start` is for.


In [ ]:
if trajs:
    fig = plot_action_usage(trajs[S0], ACTION_NAMES, n_bins=25)
    fig.suptitle(f"seed {S0} — action usage", y=1.04, fontsize=11)
    savefig(fig, FIG / "action_usage.png")
    plt.show()

    p = trajs[S0].probs
    step = trajs[S0].global_step.astype(float)
    edges = np.linspace(step.min(), step.max() + 1, 21)
    w = np.clip(np.digitize(step, edges) - 1, 0, 19)
    print("mean pi(a) over the whole run, and the minimum reached in any 5% window:")
    for a, name in enumerate(ACTION_NAMES):
        per_win = np.array([p[w == b, a].mean() if (w == b).any() else np.nan for b in range(20)])
        flag = "   <-- effectively unused" if np.nanmin(per_win) < 0.01 else ""
        print(f"  {name:14s} mean {p[:, a].mean():.4f}   min-window {np.nanmin(per_win):.4f}{flag}")
else:
    print("no trajectories recorded (run.record_trajectories = False)")

## 9. State visitation

Where the agent actually goes, split by whether it is carrying the key. In
DoorKey the grid is two rooms separated by a locked door; a policy that has
learned the task should show clear mass in the far room **only** in the
"carrying the key" panel.

This also matters for NB02: the oracle's evaluation states must come from a
region the policy actually visits.

In [ ]:
if IS_MINIGRID and trajs:
    # The packed sim_state does not record width/height and this grid is not
    # square, so the shape must be given -- plot_minigrid_visitation refuses to
    # guess rather than silently mis-binning every visit.
    fig = plot_minigrid_visitation(trajs[S0], grid_shape=GRID_SHAPE)
    fig.suptitle(f"seed {S0} — agent position visitation", y=1.04, fontsize=11)
    savefig(fig, FIG / "state_visitation.png")
    plt.show()


## 10. CHECK — checkpoints reload, and the policy actually learned

Two checks at once. A `Checkpoint` bundles the network **and** the observation
scaler, because the scaler sits between a simulator state and the network input;
saving one without the other is the classic silent-corruption bug in this kind of
pipeline. And greedy rollouts measure whether anything was learned, separately
from the stochastic training policy.

Evaluation seeds start at 900,000 so they never collide with training layouts —
if `env.layout_seeds` is set, this is therefore a *generalisation* test and will
read lower than training success by design.

In [ ]:
def greedy_eval(ck, n_episodes=50, seed0=900_000):
    env = make_env(cfg.env.env_id, cfg.env.max_episode_steps,
                   **({"fully_observable": cfg.env.fully_observable} if IS_MINIGRID else {}))
    rets, succ, lens = [], 0, []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed0 + ep)
        total, t = 0.0, 0
        while True:
            a = int(np.argmax(ck.probs(np.asarray(obs).ravel()[None, :])[0]))
            obs, r, term, trunc, _ = env.step(a)
            total += r; t += 1
            if term:
                succ += 1; break
            if trunc:
                break
        rets.append(total); lens.append(t)
    env.close()
    return float(np.mean(rets)), succ / n_episodes, float(np.mean(lens))


rows = []
for s in SEEDS:
    for p_ in list_checkpoints(seed_dir(RUN_NAME, s) / "checkpoints"):
        ck = load_checkpoint(p_)
        ret, sr, ln = greedy_eval(ck)
        rows.append({"seed": s, "frac": f"{ck.fraction:.0%}", "step": ck.global_step,
                     "greedy_return": round(ret, 4), "greedy_success": sr,
                     "greedy_length": round(ln, 1)})

FRAC_ORDER = [f"{f:.0%}" for f in cfg.run.checkpoint_fractions]
evaldf = pd.DataFrame(rows)
evaldf["frac"] = pd.Categorical(evaldf["frac"], categories=FRAC_ORDER, ordered=True)
display(evaldf.pivot(index="seed", columns="frac", values=["greedy_return", "greedy_success"]))
evaldf

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for s in SEEDS:
    sub = evaldf[evaldf.seed == s].sort_values("step")
    ax.plot(sub["step"], sub["greedy_success"], marker="o", lw=1.6, label=f"seed {s}")
ax.set_xlabel("environment steps"); ax.set_ylabel("greedy success rate (50 episodes)")
ax.set_title("Greedy policy across checkpoints")
ax.set_ylim(-0.05, 1.05); ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
savefig(fig, FIG / "greedy_checkpoints.png")
plt.show()

## 11. CHECK — trajectory files reload and are structurally sound

`dataio.trajectory.validate` asserts the properties NB02–06 depend on: `probs`
rows sum to 1 and match `logprob`; `next_obs[t] == obs[t+1]` inside an episode
(the recorded successor is the true successor, never a post-reset observation);
no row is both terminated and truncated; everything finite.

With `trajectory_stride > 1` the successor check is skipped for dropped rows, so
it validates the rows that were kept rather than the full stream.

In [ ]:
all_clean = True
for s, t in trajs.items():
    p_ = seed_dir(RUN_NAME, s) / "trajectories.npz"
    print(f"seed {s}  ({p_.stat().st_size / 1e6:.1f} MB)")
    problems = validate(t, n_actions=t.n_actions)
    print("  ->", problems if problems else "NO PROBLEMS")
    all_clean &= not problems
    print()
print("TRAJECTORY VALIDATION:", "PASS" if all_clean else "FAIL")

## 12. Gate 1 verdict

In [ ]:
# Uniform-random success, MEASURED on this env with this repo's env_pool
# (250 episodes, uniform policy). UnlockPickup: 0/250 = 0.000.
# For calibration: Unlock 0.040, DoorKey-5x5 0.072, RedBlueDoors-6x6 0.104,
# DoorKey-8x8 0.021.
RANDOM_BASELINE = {"MiniGrid-DoorKey-8x8-v0": 0.021,
                   "MiniGrid-Unlock-v0": 0.040,
                   "MiniGrid-UnlockPickup-v0": 0.000}.get(cfg.env.env_id, 0.0)

best_greedy = {s: evaldf[evaldf.seed == s]["greedy_success"].max() for s in SEEDS}
# The actions that can advance the chain are pickup, drop, and toggle. Upstream
# docs label drop unused, but MiniGrid's generic pickup logic requires an empty
# carrying slot, so a policy that drives pi(drop) to zero may never pick up the
# final box after unlocking the door.
mandatory = [3, 4, 5] if IS_MINIGRID else []

checks = {
    "every seed reaches the goal at least once":
        all(episodes[s]["success"].any() for s in SEEDS),
    f"greedy success beats uniform random ({RANDOM_BASELINE:.3f}) at some checkpoint":
        all(best_greedy[s] > RANDOM_BASELINE for s in SEEDS),
    "mandatory actions stay usable (mean pi > 0.01)":
        (not trajs) or all(all(trajs[s].probs[:, a].mean() > 0.01 for a in mandatory)
                           for s in SEEDS),
    # MEDIAN, not the last value. Once the task is solved every episode
    # returns nearly the same thing, so var(returns) -- the DENOMINATOR of
    # explained variance -- collapses and EV becomes a very noisy statistic
    # computed from one 2048-sample rollout. On the 5x5 run EV had median
    # 0.614 with 88% of updates above 0.3, and the single final update read
    # 0.19. Judging the critic on that one number is judging noise.
    "critic is informative (median explained variance > 0.3)":
        all(scalars[s]["explained_variance"].median() > 0.3 for s in SEEDS),
    "the rollouts carried signal (median adv_std_raw > 1e-4)":
        all(scalars[s]["adv_std_raw"].median() > 1e-4 for s in SEEDS),
    "trajectory files reload and validate": all_clean,
    f"{len(cfg.run.checkpoint_fractions)} checkpoints per seed, all reloadable":
        all(len(list_checkpoints(seed_dir(RUN_NAME, s) / "checkpoints"))
            == len(cfg.run.checkpoint_fractions) for s in SEEDS),
}

width = max(len(k) for k in checks)
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k:<{width}}")

print()
print("GATE 1:", "PASS — the baseline learns and the dataset is sound"
      if all(checks.values()) else "FAIL — see TUNING.md before proceeding")
print()
for s in SEEDS:
    print(f"  seed {s}: best greedy success {best_greedy[s]:.2f}, "
          f"final training success {scalars[s]['success_rate_100'].iloc[-1]:.2f}")

# The critic's error in the units of the reward, which does not degrade as the
# return variance collapses -- read this alongside EV, not instead of it.
for s in SEEDS:
    ev = scalars[s]["explained_variance"]
    rmse = float(np.sqrt(2 * scalars[s]["v_loss"].iloc[-5:].mean()))
    print(f"  seed {s}: EV median {ev.median():.3f}, {float((ev > 0.3).mean()):.0%} updates > 0.3, "
          f"final value RMSE ~{rmse:.3f} reward units")


---

## Where to go next

**If this rung passed**, the baseline is ready for the PPO-CF comparison. Create
an `unlockpickup_cf.yaml` from this config and vary only `ppo.pg_mode` across
`gae`, `cf_all_action`, and `cf_shuffled`, following the existing Unlock and
RedBlueDoors notebooks.

**If it failed**, `TUNING.md` lists the knobs in the order worth trying, each
tied to the diagnostic that justifies it. Do not tune more than one at a time,
and change them in `config/envs/<env>.yaml` (or `OVERRIDES` above) — never in
this notebook, so a result stays traceable to a committed configuration.

## What Notebook 02 inherits

Per seed, in `runs/<run_name>/seed_<n>/`:

| artifact | contents |
|---|---|
| `trajectories.npz` | one row per recorded step; `sim_state` is the packed MiniGrid state (`grid.encode()` + agent col/row/dir + carried object + step count), not the observation |
| `checkpoints/ckpt_*.pt` | weights **and** the frozen observation scaler, at 10/30/50/75/100% |
| `scalars.csv`, `episodes.csv` | diagnostics, including the sub-goal ladder |
| `../config.json` | the exact configuration used |

State restore should be verified before running a CF arm here, exactly as the
Unlock and RedBlueDoors CF notebooks do:

```python
from envs.env_pool import make_env, set_sim_state, get_sim_state
env = make_env(cfg.env.env_id, fully_observable=True)
obs = set_sim_state(env, traj.sim_state[i])    # exact restore, wrappers re-applied
```

Two things to settle before running NB02 here:

1. **Which checkpoint the oracle uses.** Check `explained_variance`, reward
   coverage, and the greedy table before picking.
2. **K = 7 changes the null.** Chance best-action agreement is 1/7 ≈ 0.143.
   `pickup`, `drop`, and `toggle` are the actions to inspect most closely.
3. **Reward shaping changes what the oracle measures.** If `reward.potential_shaping`
   was on, the critic learned `V_shaped = V_true − Φ`; add Φ back before building
   the landscape. If a count bonus was on, the checkpoint is not usable for the
   oracle at all — which is why the trainer refuses to save one while the bonus
   is still nonzero.
